# E-Commerce Intelligence System

## 01 — Data Profiling

### Objective

Understand the structure, quality, and relationships of the raw Olist e-commerce datasets before performing data cleaning, transformation, and analysis.

### Dataset

Olist Brazilian E-Commerce Public Dataset

### Phase

Phase 1 — Data Profiling

In [17]:
from pathlib import Path
import pandas as pd
import numpy as np

In [18]:
DATA_DIR = Path("../data/raw")

In [19]:
DATA_DIR.exists()

True

In [20]:
csv_files = sorted(DATA_DIR.glob("*.csv"))

print(f"Number of CSV files: {len(csv_files)}")

for file in csv_files:
    print(file.name)

Number of CSV files: 9
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [21]:
tables = {}

for file in csv_files:
    table_name = file.stem
    tables[table_name] = pd.read_csv(file)
    

In [22]:
tables


{'olist_customers_dataset':                             customer_id                customer_unique_id  \
 0      06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
 1      18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
 2      4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
 3      b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
 4      4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   
 ...                                 ...                               ...   
 99436  17ddf5dd5d51696bb3d7c6291687be6f  1a29b476fee25c95fbafc67c5ac95cf8   
 99437  e7b71a9017aa05c9a7fd292d714858e8  d52a67c98be1cf6a5c84435bd38d095d   
 99438  5e28dfe12db7fb50a4b2f691faecea5e  e9f50caf99f032f0bf3c55141f019d99   
 99439  56b18e2166679b8a959d72dd06da27f9  73c2643a0a458b49f58cea58833b192e   
 99440  274fa6071e5e17fe303b9748641082c8  84732c5050c01db9b23e19ba39899398   
 
        customer_zip_code_prefix   

In [23]:
tables.keys()


dict_keys(['olist_customers_dataset', 'olist_geolocation_dataset', 'olist_order_items_dataset', 'olist_order_payments_dataset', 'olist_order_reviews_dataset', 'olist_orders_dataset', 'olist_products_dataset', 'olist_sellers_dataset', 'product_category_name_translation'])

In [24]:
tables["olist_orders_dataset"].head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [25]:
tables.keys()

dict_keys(['olist_customers_dataset', 'olist_geolocation_dataset', 'olist_order_items_dataset', 'olist_order_payments_dataset', 'olist_order_reviews_dataset', 'olist_orders_dataset', 'olist_products_dataset', 'olist_sellers_dataset', 'product_category_name_translation'])

In [26]:
dataset_inventory = []

for table_name, df in tables.items():
    dataset_inventory.append({
        "table": table_name,
        "rows": df.shape[0],
        "columns": df.shape[1]
    })

dataset_inventory = pd.DataFrame(dataset_inventory)

dataset_inventory

,table,rows,columns
0,olist_customers_dataset,99441,5
1,olist_geolocation_dataset,1000163,5
2,olist_order_items_dataset,112650,7
3,olist_order_payments_dataset,103886,5
4,olist_order_reviews_dataset,99224,7
5,olist_orders_dataset,99441,8
6,olist_products_dataset,32951,9
7,olist_sellers_dataset,3095,4
8,product_category_name_translation,71,2


## 2. Schema and Data Quality Profiling

We inspect each table's columns, data types, missing values, and cardinality.


In [27]:
def profile_table(df, table_name):
    profile = pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "non_null": df.notna().sum().values,
        "missing": df.isna().sum().values,
        "missing_pct": (df.isna().mean() * 100).round(2).values,
        "unique": df.nunique().values
    })

    print(f"Table: {table_name}")
    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns)}")
    print(f"Duplicate rows: {df.duplicated().sum():,}")

    return profile


In [28]:
orders = tables["olist_orders_dataset"]

orders_profile = profile_table(
    orders,
    "olist_orders_dataset"
)

orders_profile

Table: olist_orders_dataset
Rows: 99,441
Columns: 8
Duplicate rows: 0


,column,dtype,non_null,missing,missing_pct,unique
0,order_id,object,99441,0,0.00,99441
1,customer_id,object,99441,0,0.00,99441
2,order_status,object,99441,0,0.00,8
3,order_purchase_timestamp,object,99441,0,0.00,98875
4,order_approved_at,object,99281,160,0.16,90733
5,order_delivered_carrier_date,object,97658,1783,1.79,81018
6,order_delivered_customer_date,object,96476,2965,2.98,95664
7,order_estimated_delivery_date,object,99441,0,0.00,459


In [29]:
profiles = {}

for table_name, df in tables.items():
    profiles[table_name] = profile_table(df, table_name)

Table: olist_customers_dataset
Rows: 99,441
Columns: 5
Duplicate rows: 0
Table: olist_geolocation_dataset
Rows: 1,000,163
Columns: 5
Duplicate rows: 261,831
Table: olist_order_items_dataset
Rows: 112,650
Columns: 7
Duplicate rows: 0
Table: olist_order_payments_dataset
Rows: 103,886
Columns: 5
Duplicate rows: 0
Table: olist_order_reviews_dataset
Rows: 99,224
Columns: 7
Duplicate rows: 0
Table: olist_orders_dataset
Rows: 99,441
Columns: 8
Duplicate rows: 0
Table: olist_products_dataset
Rows: 32,951
Columns: 9
Duplicate rows: 0
Table: olist_sellers_dataset
Rows: 3,095
Columns: 4
Duplicate rows: 0
Table: product_category_name_translation
Rows: 71
Columns: 2
Duplicate rows: 0


In [30]:
for table_name, profile in profiles.items():
    print("\n" + "=" * 80)
    print(table_name)
    display(profile)


olist_customers_dataset


,column,dtype,non_null,missing,missing_pct,unique
0,customer_id,object,99441,0,0.0,99441
1,customer_unique_id,object,99441,0,0.0,96096
2,customer_zip_code_prefix,int64,99441,0,0.0,14994
3,customer_city,object,99441,0,0.0,4119
4,customer_state,object,99441,0,0.0,27



olist_geolocation_dataset


,column,dtype,non_null,missing,missing_pct,unique
0,geolocation_zip_code_prefix,int64,1000163,0,0.0,19015
1,geolocation_lat,float64,1000163,0,0.0,717360
2,geolocation_lng,float64,1000163,0,0.0,717613
3,geolocation_city,object,1000163,0,0.0,8011
4,geolocation_state,object,1000163,0,0.0,27



olist_order_items_dataset


...

,column,dtype,non_null,missing,missing_pct,unique
0,product_id,object,32951,0,0.00,32951
1,product_category_name,object,32341,610,1.85,73
2,product_name_lenght,float64,32341,610,1.85,66
3,product_description_lenght,float64,32341,610,1.85,2960
4,product_photos_qty,float64,32341,610,1.85,19
5,product_weight_g,float64,32949,2,0.01,2204
6,product_length_cm,float64,32949,2,0.01,99
7,product_height_cm,float64,32949,2,0.01,102
8,product_width_cm,float64,32949,2,0.01,95



olist_sellers_dataset


,column,dtype,non_null,missing,missing_pct,unique
0,seller_id,object,3095,0,0.0,3095
1,seller_zip_code_prefix,int64,3095,0,0.0,2246
2,seller_city,object,3095,0,0.0,611
3,seller_state,object,3095,0,0.0,23



product_category_name_translation


,column,dtype,non_null,missing,missing_pct,unique
0,product_category_name,object,71,0,0.0,71
1,product_category_name_english,object,71,0,0.0,71


In [31]:
profiles = {}

for table_name, df in tables.items():
    profiles[table_name] = profile_table(df, table_name)

Table: olist_customers_dataset
Rows: 99,441
Columns: 5
Duplicate rows: 0
Table: olist_geolocation_dataset
Rows: 1,000,163
Columns: 5
Duplicate rows: 261,831
Table: olist_order_items_dataset
Rows: 112,650
Columns: 7
Duplicate rows: 0
Table: olist_order_payments_dataset
Rows: 103,886
Columns: 5
Duplicate rows: 0
Table: olist_order_reviews_dataset
Rows: 99,224
Columns: 7
Duplicate rows: 0
Table: olist_orders_dataset
Rows: 99,441
Columns: 8
Duplicate rows: 0
Table: olist_products_dataset
Rows: 32,951
Columns: 9
Duplicate rows: 0
Table: olist_sellers_dataset
Rows: 3,095
Columns: 4
Duplicate rows: 0
Table: product_category_name_translation
Rows: 71
Columns: 2
Duplicate rows: 0


In [32]:

for table_name, profile in profiles.items():
    print("\n" + "=" * 80)
    print(table_name)
    display(profile)


olist_customers_dataset


,column,dtype,non_null,missing,missing_pct,unique
0,customer_id,object,99441,0,0.0,99441
1,customer_unique_id,object,99441,0,0.0,96096
2,customer_zip_code_prefix,int64,99441,0,0.0,14994
3,customer_city,object,99441,0,0.0,4119
4,customer_state,object,99441,0,0.0,27



olist_geolocation_dataset


,column,dtype,non_null,missing,missing_pct,unique
0,geolocation_zip_code_prefix,int64,1000163,0,0.0,19015
1,geolocation_lat,float64,1000163,0,0.0,717360
2,geolocation_lng,float64,1000163,0,0.0,717613
3,geolocation_city,object,1000163,0,0.0,8011
4,geolocation_state,object,1000163,0,0.0,27



olist_order_items_dataset


...

,column,dtype,non_null,missing,missing_pct,unique
0,product_id,object,32951,0,0.00,32951
1,product_category_name,object,32341,610,1.85,73
2,product_name_lenght,float64,32341,610,1.85,66
3,product_description_lenght,float64,32341,610,1.85,2960
4,product_photos_qty,float64,32341,610,1.85,19
5,product_weight_g,float64,32949,2,0.01,2204
6,product_length_cm,float64,32949,2,0.01,99
7,product_height_cm,float64,32949,2,0.01,102
8,product_width_cm,float64,32949,2,0.01,95



olist_sellers_dataset


,column,dtype,non_null,missing,missing_pct,unique
0,seller_id,object,3095,0,0.0,3095
1,seller_zip_code_prefix,int64,3095,0,0.0,2246
2,seller_city,object,3095,0,0.0,611
3,seller_state,object,3095,0,0.0,23



product_category_name_translation


,column,dtype,non_null,missing,missing_pct,unique
0,product_category_name,object,71,0,0.0,71
1,product_category_name_english,object,71,0,0.0,71


# 3. Relationship and Key Validation

Before joining or transforming the datasets, we validate primary-key candidates, foreign-key relationships, and table cardinality.

In [34]:
customers = tables["olist_customers_dataset"]
orders = tables["olist_orders_dataset"]

In [35]:
print("Customer rows:", len(customers))
print("Unique customer_id:", customers["customer_id"].nunique())
print("Duplicate customer_id:", customers["customer_id"].duplicated().sum())

Customer rows: 99441
Unique customer_id: 99441
Duplicate customer_id: 0


In [36]:
print("Order rows:", len(orders))
print("Unique customer_id in orders:", orders["customer_id"].nunique())

Order rows: 99441
Unique customer_id in orders: 99441


In [37]:
orders_per_customer = (
    orders
    .groupby("customer_id")
    .size()
    .sort_values(ascending=False)
)

orders_per_customer.head(10)

customer_id
00012a2ce6f8dcda20d059ce98491703    1
000161a058600d5901f007fab4c27140    1
0001fd6190edaaf884bcaf3d49edf079    1
0002414f95344307404f0ace7a26f1d5    1
000379cdec625522490c315e70c7a9fb    1
0004164d20a9e969af783496f3408652    1
000419c5494106c306a97b5635748086    1
00046a560d407e99b969756e0b10f282    1
00050bf6e01e69d5c0fd612f1bcfb69c    1
000598caf2ef4117407665ac33275130    1
dtype: int64

In [38]:
orders_per_customer.value_counts().sort_index()

1    99441
Name: count, dtype: int64

In [39]:
customer_ids = set(customers["customer_id"])

orphan_orders = orders[
    ~orders["customer_id"].isin(customer_ids)
]

print("Orders with unknown customer_id:", len(orphan_orders))

Orders with unknown customer_id: 0


In [40]:
print("Customer rows:", len(customers))
print("Unique customer_id:", customers["customer_id"].nunique())
print("Unique customer_unique_id:", customers["customer_unique_id"].nunique())

Customer rows: 99441
Unique customer_id: 99441
Unique customer_unique_id: 96096


In [41]:
customer_record_counts = (
    customers
    .groupby("customer_unique_id")
    .size()
    .sort_values(ascending=False)
)

customer_record_counts.head(10)

customer_unique_id
8d50f5eadf50201ccdcedfb9e2ac8455    17
3e43e6105506432c953e165fb2acf44c     9
1b6c7548a2a1f9037c1fd3ddfed95f33     7
6469f99c1f9dfae7733b25662e7f1782     7
ca77025e7201e3b30c44b472ff346268     7
47c1a3033b8b77b3ab6e109eb4d5fdf3     6
12f5d6e1cbf93dafd9dcc19095df0b3d     6
63cfc61cee11cbe306bff5857d00bfe4     6
dc813062e0fc23409cd255f7f53c7074     6
de34b16117594161a6a89c50b289d35a     6
dtype: int64

In [42]:
print(
    "Customers with multiple customer_id records:",
    (customer_record_counts > 1).sum()
)

Customers with multiple customer_id records: 2997


In [43]:
relationship_summary = pd.DataFrame({
    "metric": [
        "Customer rows",
        "Unique customer_id",
        "Unique customer_unique_id",
        "Order rows",
        "Unique customer_id in orders",
        "Customers with multiple customer_id records",
        "Customers with multiple orders",
        "Orders with unknown customer_id"
    ],
    "value": [
        len(customers),
        customers["customer_id"].nunique(),
        customers["customer_unique_id"].nunique(),
        len(orders),
        orders["customer_id"].nunique(),
        (customer_record_counts > 1).sum(),
        (orders_per_customer > 1).sum(),
        len(orphan_orders)
    ]
})

relationship_summary

,metric,value
0,Customer rows,99441
1,Unique customer_id,99441
2,Unique customer_unique_id,96096
3,Order rows,99441
4,Unique customer_id in orders,99441
5,Customers with multiple customer_id records,2997
6,Customers with multiple orders,0
7,Orders with unknown customer_id,0


In [44]:
orders_with_customer = orders.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left",
    validate="one_to_one"
)

In [45]:
validate="one_to_one"

In [46]:
orders_with_customer[
    "customer_unique_id"
].nunique()

96096

In [47]:
orders_per_unique_customer = (
    orders_with_customer
    .groupby("customer_unique_id")
    .size()
    .sort_values(ascending=False)
)

orders_per_unique_customer.head(10)

customer_unique_id
8d50f5eadf50201ccdcedfb9e2ac8455    17
3e43e6105506432c953e165fb2acf44c     9
1b6c7548a2a1f9037c1fd3ddfed95f33     7
6469f99c1f9dfae7733b25662e7f1782     7
ca77025e7201e3b30c44b472ff346268     7
47c1a3033b8b77b3ab6e109eb4d5fdf3     6
12f5d6e1cbf93dafd9dcc19095df0b3d     6
63cfc61cee11cbe306bff5857d00bfe4     6
dc813062e0fc23409cd255f7f53c7074     6
de34b16117594161a6a89c50b289d35a     6
dtype: int64

In [48]:
orders_per_unique_customer.value_counts().sort_index()

1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1
Name: count, dtype: int64

In [49]:
orders_with_customer = orders.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left",
    validate="one_to_one"
)

In [50]:
orders_with_customer["customer_unique_id"].nunique()

96096

In [52]:
orders_per_unique_customer = (
    orders_with_customer
    .groupby("customer_unique_id")
    .size()
    .sort_values(ascending=False)
)

orders_per_unique_customer.value_counts().sort_index()

1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1
Name: count, dtype: int64

In [ ]:
orders = tables["olist_orders_dataset"]

orders_profile = profile_table(
    orders,
    "olist_orders_dataset"
)

orders_profile

Table: olist_orders_dataset
Rows: 99,441
Columns: 8
Duplicate rows: 0


,column,dtype,non_null,missing,missing_pct,unique
0,order_id,object,99441,0,0.00,99441
1,customer_id,object,99441,0,0.00,99441
2,order_status,object,99441,0,0.00,8
3,order_purchase_timestamp,object,99441,0,0.00,98875
4,order_approved_at,object,99281,160,0.16,90733
5,order_delivered_carrier_date,object,97658,1783,1.79,81018
6,order_delivered_customer_date,object,96476,2965,2.98,95664
7,order_estimated_delivery_date,object,99441,0,0.00,459


## Customer–Order Relationship Findings

- `customers.customer_id` is unique and can serve as the primary key for the customer-record table.
- Every `orders.customer_id` exists in the customers table; no orphan orders were found.
- `customer_id` occurs only once in the orders table, so it cannot be used to identify repeat purchasing behavior.
- `customer_unique_id` represents the underlying customer across multiple customer records.
- There are 96,096 unique `customer_unique_id` values compared with 99,441 customer records.
- Repeat purchasing must therefore be analyzed using `customer_unique_id`.
- The customer-to-order relationship at the real-customer level is one-to-many.

In [53]:
order_items = tables["olist_order_items_dataset"]

In [54]:
print("Order rows:", len(orders))
print("Order-item rows:", len(order_items))
print("Unique orders in order_items:", order_items["order_id"].nunique())

Order rows: 99441
Order-item rows: 112650
Unique orders in order_items: 98666


In [55]:
items_per_order = (
    order_items
    .groupby("order_id")
    .size()
    .sort_values(ascending=False)
)

items_per_order.head(10)

order_id
8272b63d03f5f79c56e9e4120aec44ef    21
1b15974a0141d54e36626dca3fdc731a    20
ab14fdcfbe524636d65ee38360e22ce8    20
9ef13efd6949e4573a18964dd1bbe7f5    15
428a2f660dc84138d969ccd69a0ab6d5    15
9bdc4d4c71aa1de4606060929dee888c    14
73c8ab38f07dc94389065f7eba4f297a    14
37ee401157a3a0b28c9c6d0ed8c3b24b    13
af822dacd6f5cff7376413c03a388bb7    12
3a213fcdfe7d98be74ea0dc05a8b31ae    12
dtype: int64

In [56]:
items_per_order.value_counts().sort_index()

1     88863
2      7516
3      1322
4       505
5       204
6       198
7        22
8         8
9         3
10        8
11        4
12        5
13        1
14        2
15        2
20        2
21        1
Name: count, dtype: int64

In [57]:
order_ids = set(orders["order_id"])

orphan_items = order_items[
    ~order_items["order_id"].isin(order_ids)
]

print("Order items with unknown order_id:", len(orphan_items))

Order items with unknown order_id: 0


In [58]:
orders_with_items = set(order_items["order_id"])

orders_without_items = orders[
    ~orders["order_id"].isin(orders_with_items)
]

print("Orders without order items:", len(orders_without_items))

Orders without order items: 775


In [59]:
items_per_order.value_counts().sort_index()

1     88863
2      7516
3      1322
4       505
5       204
6       198
7        22
8         8
9         3
10        8
11        4
12        5
13        1
14        2
15        2
20        2
21        1
Name: count, dtype: int64

In [60]:
order_ids = set(orders["order_id"])

orphan_items = order_items[
    ~order_items["order_id"].isin(order_ids)
]

print("Order items with unknown order_id:", len(orphan_items))

Order items with unknown order_id: 0


In [61]:
orders_with_items = set(order_items["order_id"])

orders_without_items = orders[
    ~orders["order_id"].isin(orders_with_items)
]

print("Orders without order items:", len(orders_without_items))

Orders without order items: 775


In [62]:
orders_without_items["order_status"].value_counts()

order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

In [63]:
print(
    f"Orders without items: {len(orders_without_items):,}"
)

print(
    f"Percentage of all orders: "
    f"{len(orders_without_items) / len(orders) * 100:.2f}%"
)

Orders without items: 775
Percentage of all orders: 0.78%


In [64]:
payments = tables["olist_order_payments_dataset"]

In [65]:
orders_without_items_payment_check = orders_without_items[
    ["order_id", "order_status"]
].merge(
    payments[["order_id", "payment_value", "payment_type"]],
    on="order_id",
    how="left"
)

orders_without_items_payment_check.head(20)

,order_id,order_status,payment_value,payment_type
0,8e24261a7e58791d10cb1bf9da94df5c,unavailable,84.00,credit_card
1,c272bcd21c287498b4883c7512019702,unavailable,97.68,credit_card
2,37553832a3a89c9b2db59701c357ca67,unavailable,132.46,boleto
3,d57e15fb07fd180f06ab3926b39edcd2,unavailable,134.38,boleto
4,00b1cb0320190ca0daa2c88b35206009,canceled,0.00,not_defined
5,2f634e2cebf8c0283e7ef0989f77d217,unavailable,615.53,credit_card
6,ee0db22a8e742b752914016708470ec8,unavailable,167.82,credit_card
7,ed3efbd3a87bea76c2812c66a0b32219,canceled,191.46,voucher
8,6ad57aecbae806a7e9cc2cdb6b380711,unavailable,161.47,credit_card
9,df8282afe61008dc26c6c31011474d02,canceled,139.96,boleto


In [66]:
payment_coverage = (
    orders_without_items["order_id"]
    .isin(payments["order_id"])
)

print("Orders without items:", len(orders_without_items))
print("Orders without items but with payment:", payment_coverage.sum())
print("Orders without items and without payment:", (~payment_coverage).sum())

Orders without items: 775
Orders without items but with payment: 775
Orders without items and without payment: 0


## Order–Order Item Relationship

- `orders.order_id` is unique within the orders table.
- `order_items.order_id` is not unique because an order can contain multiple items.
- 0 order-item records reference an unknown order.
- Therefore, every order-item record can be associated with a valid order.
- 775 orders do not have corresponding records in `order_items`.
- These 775 orders require further investigation by order status and payment activity before deciding whether they represent expected business cases or data-quality issues.

In [67]:
orders_without_items["order_status"].value_counts()

order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

In [68]:
print(
    f"Percentage of all orders: "
    f"{len(orders_without_items) / len(orders) * 100:.2f}%"
)

Percentage of all orders: 0.78%


In [69]:
print("Orders without items:", len(orders_without_items))
print("Orders without items but with payment:", payment_coverage.sum())
print("Orders without items and without payment:", (~payment_coverage).sum())


Orders without items: 775
Orders without items but with payment: 775
Orders without items and without payment: 0


In [70]:
orders_without_items[
    orders_without_items["order_status"].isin(["shipped", "invoiced"])
]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
23254,a68ce1686d536ca72bd2dadc4b8671e5,d7bed5fac093a4136216072abaf599d5,shipped,2016-10-05 01:47:40,2016-10-07 03:11:22,2016-11-07 16:37:37,NaN,2016-12-01 00:00:00
57591,2ce9683175cdab7d1c95bcbb3e36f478,b2d7ae0415dbbca535b5f7b38056dd1f,invoiced,2016-10-05 21:03:33,2016-10-06 07:46:39,NaN,NaN,2016-11-25 00:00:00
69926,e04f1da1f48bf2bbffcf57b9824f76e1,0d00d77134cae4c58695086ad8d85100,invoiced,2016-10-05 13:22:20,2016-10-06 15:51:38,NaN,NaN,2016-11-29 00:00:00


In [71]:
orders_without_items[
    orders_without_items["order_status"].isin(["shipped", "invoiced"])
][[
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
23254,a68ce1686d536ca72bd2dadc4b8671e5,d7bed5fac093a4136216072abaf599d5,shipped,2016-10-05 01:47:40,2016-10-07 03:11:22,2016-11-07 16:37:37,NaN,2016-12-01 00:00:00
57591,2ce9683175cdab7d1c95bcbb3e36f478,b2d7ae0415dbbca535b5f7b38056dd1f,invoiced,2016-10-05 21:03:33,2016-10-06 07:46:39,NaN,NaN,2016-11-25 00:00:00
69926,e04f1da1f48bf2bbffcf57b9824f76e1,0d00d77134cae4c58695086ad8d85100,invoiced,2016-10-05 13:22:20,2016-10-06 15:51:38,NaN,NaN,2016-11-29 00:00:00


In [72]:
payments[
    payments["order_id"].isin(
        orders_without_items[
            orders_without_items["order_status"].isin(
                ["shipped", "invoiced"]
            )
        ]["order_id"]
    )
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
75617,2ce9683175cdab7d1c95bcbb3e36f478,1,boleto,1,73.04
88900,a68ce1686d536ca72bd2dadc4b8671e5,1,boleto,1,77.73
99415,e04f1da1f48bf2bbffcf57b9824f76e1,1,credit_card,7,76.19


In [73]:
products = tables["olist_products_dataset"]

In [74]:
print("Order-item rows:", len(order_items))
print("Unique product_id in order_items:", order_items["product_id"].nunique())

print("Product rows:", len(products))
print("Unique product_id in products:", products["product_id"].nunique())

Order-item rows: 112650
Unique product_id in order_items: 32951
Product rows: 32951
Unique product_id in products: 32951


In [75]:
product_ids = set(products["product_id"])

orphan_product_items = order_items[
    ~order_items["product_id"].isin(product_ids)
]

print(
    "Order items with unknown product_id:",
    len(orphan_product_items)
)

Order items with unknown product_id: 0


In [76]:
sellers = tables["olist_sellers_dataset"]

In [77]:
print("Seller rows:", len(sellers))
print("Unique seller_id:", sellers["seller_id"].nunique())
print("Unique seller_id in order_items:", order_items["seller_id"].nunique())

Seller rows: 3095
Unique seller_id: 3095
Unique seller_id in order_items: 3095


In [79]:
seller_ids = set(sellers["seller_id"])

orphan_seller_items = order_items[
    ~order_items["seller_id"].isin(seller_ids)
]

print(
    "Order items with unknown seller_id:",
    len(orphan_seller_items)
)

Order items with unknown seller_id: 0


## Order Item Relationships

### Order → Order Items

- `orders.order_id` is unique in the orders table.
- `order_items.order_id` is non-unique because one order can contain multiple items.
- 0 order items reference an unknown order.
- 775 orders have no order-item records.
- The 775 orders represent 0.78% of all orders and are predominantly `unavailable` or `canceled`.
- These orders are retained and will be handled according to the analytical use case.

### Order Item → Product

- Every `order_items.product_id` exists in `products.product_id`.
- No orphan product references were found.
- `products.product_id` is unique in the product table.

### Order Item → Seller

- Every `order_items.seller_id` exists in `sellers.seller_id`.
- No orphan seller references were found.
- `sellers.seller_id` is unique in the seller table.

In [80]:
order_items["order_item_id"].value_counts().sort_index()

order_item_id
1     98666
2      9803
3      2287
4       965
5       460
6       256
7        58
8        36
9        28
10       25
11       17
12       13
13        8
14        7
15        5
16        3
17        3
18        3
19        3
20        3
21        1
Name: count, dtype: int64

In [81]:
order_items.duplicated(
    subset=["order_id", "order_item_id"]
).sum()

np.int64(0)

In [82]:
duplicate_order_items = order_items.duplicated(
    subset=["order_id", "order_item_id"]
).sum()

print(
    "Duplicate (order_id, order_item_id) combinations:",
    duplicate_order_items
)

Duplicate (order_id, order_item_id) combinations: 0


In [83]:
print(
    "Unique order_id + order_item_id combinations:",
    order_items[["order_id", "order_item_id"]].drop_duplicates().shape[0]
)

print(
    "Total order-item rows:",
    len(order_items)
)

Unique order_id + order_item_id combinations: 112650
Total order-item rows: 112650


In [84]:
order_items["order_item_id"].value_counts()

order_item_id
1     98666
2      9803
3      2287
4       965
5       460
6       256
7        58
8        36
9        28
10       25
11       17
12       13
13        8
14        7
15        5
16        3
17        3
18        3
19        3
20        3
21        1
Name: count, dtype: int64

In [85]:
duplicate_order_items = order_items.duplicated(
    subset=["order_id", "order_item_id"]
).sum()

print(
    "Duplicate (order_id, order_item_id) combinations:",
    duplicate_order_items
)

Duplicate (order_id, order_item_id) combinations: 0


In [86]:
print(
    "Total order-item rows:",
    len(order_items)
)

print(
    "Unique (order_id, order_item_id) combinations:",
    order_items[
        ["order_id", "order_item_id"]
    ].drop_duplicates().shape[0]
)

Total order-item rows: 112650
Unique (order_id, order_item_id) combinations: 112650
